# Lab 5B — Two Identities, Two Tool Lists

[Lab 5A](#) granted the agent's service principal `EXECUTE` on **one** function and
nothing else.

This lab asks the managed MCP server the same question as two different identities
and compares the answers. The result is the most important governance idea in the
course: **the tool list itself is filtered by your grants.**

| Cell group | What you do |
|---|---|
| 1–2 | Discover tools as **yourself**. |
| 3 | Get a token for the **restricted service principal**. |
| 4 | Discover tools as **it**. |
| 5 | Call the tool it was never granted, and read the error carefully. |
| 6 | Why this beats a permission check at call time. |

**Compute:** Serverless. **Prerequisite:** [Lab 5A](#) — the grants.

## 1. The managed MCP server

Databricks exposes every Unity Catalog function in a schema as an MCP tool at:

```
  /api/2.0/mcp/functions/{catalog}/{schema}
```

It speaks JSON-RPC. Two methods matter: `tools/list` and `tools/call`.

In [0]:
import json, requests
from databricks.sdk import WorkspaceClient

CATALOG, SCHEMA = "agents_labs", "retail"
w = WorkspaceClient()
HOST = w.config.host
MCP = f"{HOST}/api/2.0/mcp/functions/{CATALOG}/{SCHEMA}"


def rpc(token: str, method: str, params: dict | None = None) -> dict:
    r = requests.post(
        MCP,
        headers={"Authorization": f"Bearer {token}",
                 "Content-Type": "application/json",
                 "Accept": "application/json, text/event-stream"},
        json={"jsonrpc": "2.0", "id": 1, "method": method, "params": params or {}},
        timeout=90)
    if r.status_code != 200:
        return {"http_error": r.status_code, "body": r.text[:300]}
    return r.json()


def tool_names(token: str) -> list[str]:
    return [t["name"] for t in rpc(token, "tools/list").get("result", {}).get("tools", [])]


print(f"  MCP server: {MCP}")

  MCP server: https://adb-7405616584960877.17.azuredatabricks.net/api/2.0/mcp/functions/agents_labs/retail


## 2. Discover as yourself

In [0]:
MY_TOKEN = w.config.authenticate()["Authorization"].removeprefix("Bearer ")
me = w.current_user.me().user_name

mine = tool_names(MY_TOKEN)
print(f"  as {me}\n  tools discovered: {len(mine)}")
for t in mine:
    print(f"    - {t}")

  as admin@datacouchoutlook.onmicrosoft.com
  tools discovered: 2
    - agents_labs__retail__get_order_summary
    - agents_labs__retail__revenue_by_region


## 3. Become the restricted identity

The agent does not run as you. It runs as a service principal that was granted
`EXECUTE` on exactly one function in [Lab 5A](#).

Its credentials live in a **Databricks secret scope**, not in this notebook.
`dbutils.secrets.get` returns the value and **redacts it from every output**, so you
cannot print it by accident — try it and see.

In [0]:
SCOPE = "agents-labs"

try:
    CLIENT_ID = dbutils.secrets.get(scope=SCOPE, key="restricted_client_id")
    CLIENT_SECRET = dbutils.secrets.get(scope=SCOPE, key="restricted_client_secret")
except Exception as e:
    raise RuntimeError(
        f"Secret scope {SCOPE!r} is not set up. Your instructor creates it with:\n"
        f"  databricks secrets create-scope {SCOPE}\n"
        f"  databricks secrets put-secret {SCOPE} restricted_client_id\n"
        f"  databricks secrets put-secret {SCOPE} restricted_client_secret\n"
        f"Original error: {e}")

# Watch what happens when we try to print it.
print(f"  client_id : {CLIENT_ID}")
print(f"  secret    : {CLIENT_SECRET}")

# OAuth M2M: exchange client credentials for a short-lived token.
tok = requests.post(
    f"{HOST}/oidc/v1/token",
    auth=(CLIENT_ID, CLIENT_SECRET),
    data={"grant_type": "client_credentials", "scope": "all-apis"},
    timeout=60).json()
SP_TOKEN = tok["access_token"]
print(f"\n  got an SP token, expires in {tok.get('expires_in')}s")

  client_id : [REDACTED]
  secret    : [REDACTED]

  got an SP token, expires in 3600s


## 4. Discover as the service principal

Same server. Same request. Different bearer token.

In [0]:
theirs = tool_names(SP_TOKEN)

print(f"  as the service principal\n  tools discovered: {len(theirs)}")
for t in theirs:
    print(f"    - {t}")

print(f"\n  you see    : {len(mine)}")
print(f"  it sees    : {len(theirs)}")
print(f"  invisible  : {sorted(set(mine) - set(theirs))}")

  as the service principal
  tools discovered: 1
    - agents_labs__retail__get_order_summary

  you see    : 2
  it sees    : 1
  invisible  : ['agents_labs__retail__revenue_by_region']


## 5. Call the tool it was never granted

`revenue_by_region` is in the list for you and absent from the list for the service
principal. Ask it to run anyway, and **read the error message**.

In [0]:
def call(token: str, fn: str, args: dict) -> str:
    res = rpc(token, "tools/call",
              {"name": f"{CATALOG}__{SCHEMA}__{fn}", "arguments": args})
    if "error" in res:
        return f"ERROR   {res['error']}"
    content = res.get("result", {}).get("content", [{}])
    return "OK      " + str(content[0].get("text", ""))[:120]


print("  as you:")
print("   get_order_summary  ", call(MY_TOKEN, "get_order_summary", {"order_ref": "ORD-1044"}))
print("   revenue_by_region  ", call(MY_TOKEN, "revenue_by_region", {"from_month": "2026-08"}))

print("\n  as the service principal:")
print("   get_order_summary  ", call(SP_TOKEN, "get_order_summary", {"order_ref": "ORD-1044"}))
print("   revenue_by_region  ", call(SP_TOKEN, "revenue_by_region", {"from_month": "2026-08"}))

  as you:
   get_order_summary   OK      {"is_truncated":false,"columns":["order_id","status","item","units","revenue","order_date","region","tier"],"rows":[["OR
   revenue_by_region   OK      {"is_truncated":false,"columns":["region","month","revenue","orders"],"rows":[["Benelux","2026-09",2112.0,1],["UK","2026

  as the service principal:
   get_order_summary   OK      {"is_truncated":false,"columns":["order_id","status","item","units","revenue","order_date","region","tier"],"rows":[["OR
   revenue_by_region   ERROR   {'code': -32602, 'message': "BAD_REQUEST: Function 'agents_labs.retail.revenue_by_region' not found"}


### Read that last error again

```
  BAD_REQUEST: Function 'agents_labs.retail.revenue_by_region' not found
```

**Not "permission denied". Not "forbidden". Not found.**

To an identity without the grant, that function does not exist. Unity Catalog is not
refusing the call — it is answering honestly about a namespace that, for this
principal, contains one function.

## 6. Why filtered discovery beats a check at call time

The usual design is: show the agent every tool, then check permissions when it calls
one. Compare:

| | Check at call time | **Filtered discovery** |
|---|---|---|
| What the model sees | every tool | only its own |
| Failure mode | tries, gets denied, retries, apologises to the user | never considers it |
| Prompt-injection surface | *"call the admin tool"* is a thing it can attempt | the tool is not in its context |
| Leak | the tool's **name and description** disclose that it exists | nothing |
| Where the rule lives | your application code | **the grant** |

> 🚨 **The tool list is context, and context leaks.** A tool called
> `approve_refund_above_limit` tells a determined user something about your system
> even if every call is refused. Filtered discovery means there is nothing to tell.

> 💡 **And notice where the rule lives.** You wrote no permission code in this
> notebook. The boundary is a `GRANT`, and it is inherited by **every** consumer of
> that schema — this notebook, an agent, a job, a colleague's app.
>
> That is the same contrast as [Lab 4B](#): the Genie Knowledge Store travelled to a
> new consumer because it lived with the agent, while [Lab 3A](#)'s `audience` filter
> did not travel, because it lived in the caller. **A grant is the strongest version
> of that idea** — it is enforced by the platform, not by anyone remembering.

### Try this before you move on

1. Grant the SP `EXECUTE` on `revenue_by_region`, re-run cell 4, and watch the tool
   appear. Revoke it and watch it vanish. **No redeploy.**
2. Try `SELECT * FROM agents_labs.retail.orders` as the SP. It has no `SELECT` — but
   `get_order_summary` reads that table and works. That is definer's rights.
3. Look again at cell 3's output. Did the secret print?